# 1.Feature Engineering

### (REVIEW)

Let's go over the description of what I intend to do in my project from what I said in deliverable 1:

### 1.1 Description of the problem

I want to investigate the accuracy and efficiency of Convolutional Neural Networks (CNNs) for autoregressive text generation. For this project, I want to train and test different CNN models by changing their parameters and embedding dimensions to see how well they can generate text. I also want to experiment with different levels of text preprocessing to see how this affects the models performance and ability to generate coherent text.

I will then compare the results using metrics such as tokens per second, perplexity, model size and cross-entropy loss to evaluate how well the different models perform.

I am going to use the 'WikiText-103' dataset, which is raw text, This dataset comes from huggingface:

https://huggingface.co/datasets/Salesforce/wikitext?utm_source=chatgpt.com

This file 'wikitext-103-v1' is much bigger at size - 738.27 MB, my MacBook M1 chip would struggle to process so we'll use a runpod GPU

I understand the assigned says:

Feature Engineering (30 points)
 - Text representation techniques (TF-IDF, Word2Vec, FastText, BERT embeddings, etc.) (WE'RE GOING TO BUILD OUR OWN VECTOR EMBEDDING)
 - Justification for the chosen representation (WE WANT TO INVESTIGATE THE AUTO GENERATIVE ABILITY OF CNNs MODELS)

Model Selection and Training (50 points)
 - Comparison of different models (e.g., Naïve Bayes, SVMs, CNNs, LSTMs, Transformers) (WE WILL COMPARE CNN AGAINST CNN BUT ALSO AGAINST LLM)
 - Justification for the selected approach
 - Model training implementation

Evaluation and Results (45 points)
 - Performance metrics (accuracy, precision, recall, F1-score, etc.)
 - Comparison of model performance
 - Discussion of findings

In [20]:
import torch
from datasets import load_dataset

print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    dataset = load_dataset("Salesforce/wikitext","wikitext-103-raw-v1")
else:
    dataset = load_dataset("Salesforce/wikitext","wikitext-2-raw-v1")

GPU: False


In [21]:
train = dataset["train"]
test = dataset["test"]

In [22]:
import random

num = random.randint(0,len(train)-1)

print(train[num])

{'text': ''}


In [23]:
df_train = train.to_pandas()

In [24]:
all_text = " ".join(df_train["text"].dropna().astype(str))

In [25]:
all_text[:400]

'  = Valkyria Chronicles III = \n   Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . E'

In [26]:
import string
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

from torch.utils.data import TensorDataset, DataLoader

# ============== SET GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("We use:", torch.cuda.get_device_name(0) if device.type == "cuda" else "My MAC")

# ============== TOKENISE TEXT

all_text = " ".join(df_train["text"].dropna().astype(str))

title_tokens = all_text.lower().split()

# ============== REMOVE STANDALONE PUNCTUATION

title_tokens = [word for word in title_tokens if word not in string.punctuation]

# ============== COLLECT VOCABULARY

VOCAB = set(title_tokens)

VOCAB.add("<UNK>")

vocab_tokens = {word: i for i, word in enumerate(sorted(VOCAB))}

print("Vocabulary size:", len(vocab_tokens))

# ============== SPLIT INTO GROUPS OF 500 TOKENS

processed_text = []

for i in range(0, len(title_tokens), 500):

    token_group = title_tokens[i:i + 500]

    if len(token_group) == 500:
        processed_text.append(token_group)

# ============== CREATE NEW DATAFRAME

df = pd.DataFrame({"processed_text": processed_text})

We use: My MAC
Vocabulary size: 66620


In [27]:
df.head()

,processed_text
0,"[valkyria, chronicles, iii, senjō, no, valkyri..."
1,"[a, top, @-@, down, perspective, of, the, batt..."
2,"[gallia, lose, face, in, the, war, while, at, ..."
3,"[true, sequel, while, valkyria, chronicles, ii..."
4,"[music, he, felt, that, he, needed, to, incorp..."


In [28]:
vocab_tokens

{"'a": 0,
 "'aaj": 1,
 "'aak": 2,
 "'aaknal": 3,
 "'académie": 4,
 "'action": 5,
 "'ad": 6,
 "'adanle": 7,
 "'addario": 8,
 "'aeback": 9,
 "'affaires": 10,
 "'africaine": 11,
 "'ah": 12,
 "'ai": 13,
 "'ak": 14,
 "'akov": 15,
 "'al": 16,
 "'albert": 17,
 "'ale": 18,
 "'alesandro": 19,
 "'alkan": 20,
 "'amore": 21,
 "'amour": 22,
 "'an": 23,
 "'anan": 24,
 "'andrade": 25,
 "'ani": 26,
 "'anjou": 27,
 "'anshan": 28,
 "'apollon": 29,
 "'aquin": 30,
 "'aragona": 31,
 "'architecture": 32,
 "'aretz": 33,
 "'ari": 34,
 "'art": 35,
 "'artus": 36,
 "'asiyot": 37,
 "'assomption": 38,
 "'at": 39,
 "'atun": 40,
 "'aubeterre": 41,
 "'aubigny": 42,
 "'aude": 43,
 "'auteroche": 44,
 "'automobile": 45,
 "'autrui": 46,
 "'avria": 47,
 "'awiil": 48,
 "'ayab": 49,
 "'azzam": 50,
 "'brien": 51,
 "'callaghan": 52,
 "'carmel": 53,
 "'casey": 54,
 "'chard": 55,
 "'clock": 56,
 "'connell": 57,
 "'connor": 58,
 "'cruz": 59,
 "'d": 60,
 "'daniel": 61,
 "'day": 62,
 "'dell": 63,
 "'domo": 64,
 "'donnell": 65,
 "'

In [29]:
import random

num = random.randint(0,len(df)-1)

print(df["processed_text"].iloc[num])

['front', 'in', 'a', 'relatively', 'static', 'role', 'facing', 'v', 'corps', 'was', 'the', '1st', 'parachute', 'division', '1', 'fallschirmjägerdivision', 'under', 'brigadier', 'general', 'generalmajor', 'richard', 'heidrich', 'on', 'the', 'coast', 'to', 'their', 'right', 'stood', 'the', '90th', 'panzergrenadier', 'division', '90', 'panzergrenadierdivision', 'under', 'major', 'general', 'carl', '@-@', 'hans', 'lungershausen', 'succeeded', 'by', 'colonel', 'oberst', 'ernst', '@-@', 'günther', 'baade', 'on', '20', 'december', 'and', 'further', 'inland', 'of', 'them', 'was', 'the', '26th', 'panzer', 'division', '26', 'panzerdivision', 'under', 'brigadier', 'general', 'smilo', 'freiherr', 'von', 'lüttwitz', 'with', 'their', 'right', 'flank', 'on', 'orsogna', 'further', 'inland', 'facing', 'xiii', 'corps', 'was', 'the', '65th', 'infantry', 'division', '65', 'infantriedivision', 'under', 'brigadier', 'general', 'hellmuth', 'pfeifer', 'supported', 'by', 'elements', 'of', '1st', 'parachute', '

In [30]:
VOCAB

{'movies',
 'mathematical',
 'mah',
 'bushfire',
 'bandcamp',
 'intercepts',
 'ihl',
 'surround',
 'giants',
 'northumberland',
 'insb',
 'input',
 'baritone',
 'runnings',
 'arbeiderbevægelsen',
 'smathers',
 'showcases',
 'divinities',
 'mitochondria',
 'tú',
 'ayyubids',
 'cronenberg',
 'satisfying',
 'radionics',
 'sanctioned',
 'ornithology',
 'beverage',
 'semen',
 'infinity',
 'pamela',
 'financially',
 'accompaniment',
 'mernefu',
 'possibles',
 'smacked',
 'anthologist',
 '270',
 'langhelle',
 'pharmacy',
 'esculenta',
 'swept',
 'fabricated',
 'season',
 'defrauding',
 'cda67624',
 'mediocre',
 'untukmu',
 'immunoglobulins',
 'stalls',
 'numerical',
 'sirach',
 'hedgerows',
 'spore',
 'nau',
 'capella',
 'brunswick',
 'redmond',
 'si',
 'takeaway',
 'representations',
 'rookies',
 'extrude',
 'llwyd',
 'pottsii',
 'aucklands',
 'diverge',
 '991',
 '1059',
 'nereo',
 'eugenie',
 'onomatopoetic',
 'replicate',
 'unwanted',
 'ottoman',
 'embark',
 'khac',
 'lazarus',
 '30s',
 'f

In [31]:
# ============== CREATE TOKEN IDS

unk_id = vocab_tokens["<UNK>"]

df["token_ids"] = df["processed_text"].apply(lambda batch: [vocab_tokens.get(word, unk_id) for word in batch])

# ============== CREATE TENSOR

token_tensor = torch.tensor(df["token_ids"].tolist(), dtype=torch.long)

print("Tensor shape:", token_tensor.shape)

Tensor shape: torch.Size([3559, 500])


In [32]:
df.head()

,processed_text,token_ids
0,"[valkyria, chronicles, iii, senjō, no, valkyri...","[62472, 12861, 29491, 53096, 41262, 62472, 185..."
1,"[a, top, @-@, down, perspective, of, the, batt...","[2935, 59870, 2931, 18988, 44654, 42084, 59070..."
2,"[gallia, lose, face, in, the, war, while, at, ...","[24559, 35694, 22071, 29849, 59070, 63736, 642..."
3,"[true, sequel, while, valkyria, chronicles, ii...","[60742, 53175, 64284, 62472, 12861, 29488, 267..."
4,"[music, he, felt, that, he, needed, to, incorp...","[39978, 27488, 22605, 59063, 27488, 40685, 596..."


In [33]:
import random

num = random.randint(0,len(df)-1)

print("Length: ",len(df["token_ids"].iloc[num]), " Tokens: ",df["token_ids"].iloc[num])

Length:  500  Tokens:  [63818, 42593, 10710, 59070, 62590, 61803, 57203, 6117, 24977, 33916, 14320, 42496, 12119, 6117, 59183, 11049, 59672, 7861, 33386, 59672, 59241, 16559, 59070, 16509, 42084, 1152, 42025, 31205, 55185, 42084, 2935, 28414, 23578, 59070, 32250, 43900, 29849, 5839, 11906, 6117, 19624, 16595, 33900, 34510, 42496, 35797, 16559, 64240, 64941, 23518, 63818, 53814, 4950, 40637, 61747, 42084, 59070, 64110, 14469, 4950, 45784, 50646, 56118, 32100, 44587, 37039, 21445, 17469, 21445, 156, 12642, 29520, 4950, 27405, 4857, 61430, 49893, 44587, 63818, 29849, 35744, 64693, 21445, 4950, 27845, 35790, 17655, 59672, 28143, 21481, 64284, 27488, 26728, 8015, 61512, 6008, 60009, 28143, 16842, 4950, 54966, 27488, 37039, 27845, 47255, 2935, 50034, 53591, 26728, 40918, 33386, 21445, 4950, 32100, 64143, 37039, 18231, 29849, 2935, 13093, 12057, 29849, 32225, 42307, 1272, 42025, 1468, 4950, 29849, 2935, 12902, 64030, 42307, 2793, 16714, 1468, 1469, 46600, 20099, 62981, 3867, 28229, 49500, 241

In [34]:
print(token_tensor)

tensor([[62472, 12861, 29491,  ..., 19658, 61810, 62283],
        [ 2935, 59870,  2931,  ..., 64941, 42803, 36486],
        [24559, 35694, 22071,  ..., 59070, 53219, 23041],
        ...,
        [38266, 44458, 65341,  ..., 11789, 46813, 13970],
        [56143, 37400,  7861,  ..., 13959,  2935, 24357],
        [ 2935,  2063,  2930,  ..., 32304,  2281, 38415]])


In [35]:
len(df)

3559

In [36]:
embedding = nn.Embedding(num_embeddings=len(vocab_tokens),embedding_dim=200).to(device)

In [37]:
embedding

Embedding(66620, 200)

In [38]:
print(len(VOCAB))

66620


# 2.Model Selection and Training (50 points) (SMALL MODEL)

In [ ]:
class CNN_SIMPLE(nn.Module):

    def __init__(self, embedding, vocab_size, kernel_size=3):
        super().__init__()

        self.embedding = embedding

        self.conv = nn.Conv1d(200,128,kernel_size)

        self.fc = nn.Linear(128,vocab_size)

        self.kernel_size = kernel_size

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        x = F.pad(x, (self.kernel_size - 1, 0))

        x = torch.relu(self.conv(x))

        x = x.transpose(1, 2)

        x = self.fc(x)

        return x

model_cnn_one = CNN_SIMPLE(embedding=embedding,vocab_size=len(vocab_tokens)).to(device)

criterion = nn.CrossEntropyLoss()

x = token_tensor[:, :-1]
y = token_tensor[:, 1:]

train_dataset = TensorDataset(x, y)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

optimizer = torch.optim.Adam(model_cnn_one.parameters(), lr=0.001)


epochs = 10

for epoch in range(epochs):

    model_cnn_one.train()

    for i, (inputs, labels) in enumerate(train_loader):

        optimizer.zero_grad()

        outputs = model_cnn_one(inputs)

        loss = criterion(outputs.reshape(-1, len(vocab_tokens)),labels.reshape(-1))

        loss.backward()

        optimizer.step()

        if torch.cuda.is_available():
            if (i + 1) % 100000 == 0:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {loss.item():.4f}")

        else:
            if (i + 1) % 50 == 0:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {loss.item():.4f}")
                #torch.save({"model_state_dict": model_cnn_one.state_dict(),"vocab_tokens": vocab_tokens,}, "/Users/benjaminbrooke/PycharmProjects/Agentic/MODEL")

In [ ]:
torch.save({'model_state_dict': model_cnn_one.state_dict(),'vocab_tokens': vocab_tokens}, 'cnn_language_model.pth')

# 2.Model Selection and Training (50 points) (BIG MODEL)

In [ ]:
class BIGBlock(nn.Module):

    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()

        self.dilation = dilation
        self.kernel_size = kernel_size

        self.conv = nn.Conv1d(channels,channels * 2,kernel_size=kernel_size,dilation=dilation)

        self.residual_projection = nn.Conv1d(channels, channels, kernel_size=1)

        self.skip_projection = nn.Conv1d(channels, channels, kernel_size=1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        padding = self.dilation * (self.kernel_size - 1)

        x = F.pad(x, (padding, 0))

        x = self.conv(x)

        tanh_part, sigmoid_part = x.chunk(2, dim=1)

        x = torch.tanh(tanh_part) * torch.sigmoid(sigmoid_part)

        x = self.dropout(x)

        residual = self.residual_projection(residual)

        skip = self.skip_projection(x)

        return residual + x, skip

In [ ]:
class CNN_LANGUAGE_MODEL(nn.Module):

    def __init__(self,embedding,channels=512,kernel_size=3,dilations=None,dropout=0.1):
        super().__init__()

        self.embedding = embedding

        self.input_projection = nn.Conv1d(embedding.embedding_dim,channels,kernel_size=1)

        if dilations is None:
            dilations = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512] * 2

        self.blocks = nn.ModuleList([
            BIGBlock(channels=channels,kernel_size=kernel_size,dilation=d,dropout=dropout)for d in dilations])

        self.output_projection = nn.Conv1d(channels,embedding.embedding_dim,kernel_size=1)

        self.output_bias = nn.Parameter(torch.zeros(embedding.num_embeddings))

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        x = self.input_projection(x)

        skip_connections = []

        for block in self.blocks:
            x, skip = block(x)
            skip_connections.append(skip)

        x = torch.stack(skip_connections).sum(dim=0)

        x = x / (len(self.blocks) ** 0.5)

        x = torch.relu(x)

        x = self.output_projection(x)

        x = x.transpose(1, 2)

        logits = F.linear(x,self.embedding.weight,self.output_bias)

        return logits

In [ ]:
embedding = nn.Embedding(num_embeddings=len(vocab_tokens),embedding_dim=512).to(device)

model = CNN_LANGUAGE_MODEL(embedding=embedding,channels=512,kernel_size=3,dropout=0.1).to(device)

x = token_tensor[:, :-1]
y = token_tensor[:, 1:]

train_dataset = TensorDataset(x, y)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs.reshape(-1, len(vocab_tokens)),labels.reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, "f"Average Loss: {total_loss / len(train_loader)}")

In [ ]:
torch.save({'model_state_dict': model,'vocab_tokens': vocab_tokens}, 'LARGE_cnn.pth')

# 3.Test the Model

In [ ]:
def generate_text(model, prompt, vocab_tokens, max_new_tokens=50,
                  temperature=0.8, top_k=20):

    model.eval()

    device = next(model.parameters()).device

    id_to_word = {idx: word for word, idx in vocab_tokens.items()}

    words = prompt.lower().split()

    token_ids = [vocab_tokens.get(word, vocab_tokens['<UNK>'])for word in words]

    generated = torch.tensor([token_ids],dtype=torch.long,device=device)

    with torch.no_grad():

        for _ in range(max_new_tokens):
            context = generated[:, -500:]

            logits = model(context)

            next_token_logits = logits[:, -1, :]

            next_token_logits = next_token_logits / temperature

            values, indices = torch.topk(next_token_logits,top_k)

            probabilities = torch.softmax(values, dim=-1)

            sampled_index = torch.multinomial(probabilities,num_samples=1)

            next_token = indices.gather(-1,sampled_index)

            generated = torch.cat([generated, next_token],dim=1)

    output_words = [id_to_word[idx]for idx in generated[0].tolist()]

    return " ".join(output_words)